# 比赛预处理

**摘要。** 先看原始表，再按缺失与可用性决定清洗，写出 `cleaned.csv`。

**输入**
- `data/raw/Matches.csv`：比赛、比分、Elo、Form、赔率等
- `data/raw/EloRatings.csv`：按日期的球队 Elo

**输出**
- `data/processed/cleaned.csv`

**分析顺序**
1. 读取原始表，计算各列缺失率。
2. Matches 中 `HomeShots` 到 `C_PHB` 缺失过多，直接删列。
3. `FTHome` / `FTAway` / `FTResult` 同一批缺失行，删行。
4. 部分比赛 Elo 缺失，用 EloRatings 补全。
5. 由赛果推出 Form 公式，再重算。
6. 检查 `OddHome` / `OddDraw` / `OddAway` 异常值。
7. 双方近 5 场不完整的比赛删除。


## 1. 数据分析

1. 打印数据基础信息
2. 分析数据项缺失率

In [1]:
from pathlib import Path
from collections import defaultdict, deque
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data/raw/Matches.csv').exists():
    ROOT = ROOT.parent

pd.set_option('display.max_columns', None)
CORE = ['Division', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FTHome', 'FTAway', 'FTResult']
ODDS = ['OddHome', 'OddDraw', 'OddAway']
COUNTRY = {d: d for d in 'ARG AUT BRA CHN DEN FIN IRL JAP MEX NOR POL ROM RUS SUI SWE USA'.split()}
COUNTRY.update({d: c for c, ds in {
    'ENG': 'E0 E1 E2 E3 EC', 'SCO': 'SC0 SC1 SC2 SC3', 'GER': 'D1 D2',
    'FRA': 'F1 F2', 'ITA': 'I1 I2', 'ESP': 'SP1 SP2', 'BEL': 'B1',
    'GRE': 'G1', 'NED': 'N1', 'POR': 'P1', 'TUR': 'T1'
}.items() for d in ds.split()})

def norm(x):
    return ' '.join(unicodedata.normalize('NFKC', str(x)).split()) if pd.notna(x) else x

raw = pd.read_csv(ROOT / 'data/raw/Matches.csv')
elo = pd.read_csv(ROOT / 'data/raw/EloRatings.csv')
n_raw = len(raw)

print('Matches', raw.shape, '| EloRatings', elo.shape)
miss = pd.DataFrame({
    'n_missing': raw.isna().sum(),
    'missing_rate': raw.isna().mean(),
})
miss['missing_pct'] = (100 * miss.missing_rate).round(2)
print('\nMatches missing rate = n_missing / n_rows')
print(miss.to_string())
print('\nEloRatings missing')
print(elo.isna().sum().to_string())
display(raw.head())
display(elo.head())


C:\Users\zhong\AppData\Local\Temp\ipykernel_14376\4060780185.py:25: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(ROOT / 'data/raw/Matches.csv')


Matches (238858, 48) | EloRatings (273972, 4)

Matches missing rate = n_missing / n_rows
             n_missing  missing_rate  missing_pct
Division             0      0.000000         0.00
MatchDate            0      0.000000         0.00
MatchTime       131485      0.550474        55.05
HomeTeam             0      0.000000         0.00
AwayTeam             0      0.000000         0.00
HomeElo          88960      0.372439        37.24
AwayElo          89029      0.372728        37.27
Form3Home         1524      0.006380         0.64
Form5Home         1535      0.006426         0.64
Form3Away         1523      0.006376         0.64
Form5Away         1534      0.006422         0.64
FTHome               4      0.000017         0.00
FTAway               4      0.000017         0.00
FTResult             4      0.000017         0.00
HTHome           54580      0.228504        22.85
HTAway           54580      0.228504        22.85
HTResult         54580      0.228504        22.85
HomeShots  

,Division,MatchDate,MatchTime,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult,HTHome,HTAway,HTResult,HomeShots,AwayShots,HomeTarget,AwayTarget,HomeFouls,AwayFouls,HomeCorners,AwayCorners,HomeYellow,AwayYellow,HomeRed,AwayRed,OddHome,OddDraw,OddAway,MaxHome,MaxDraw,MaxAway,Over25,Under25,MaxOver25,MaxUnder25,HandiSize,HandiHome,HandiAway,C_LTH,C_LTA,C_VHD,C_VAD,C_HTB,C_PHB
0,F1,2000-07-28,NaN,Marseille,Troyes,1686.34,1586.57,0.0,0.0,0.0,0.0,3.0,1.0,H,2.0,1.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.65,3.3,4.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,F1,2000-07-28,NaN,Paris SG,Strasbourg,1714.89,1642.51,0.0,0.0,0.0,0.0,3.0,1.0,H,1.0,1.0,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.60,3.4,4.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F2,2000-07-28,NaN,Wasquehal,Nancy,1465.08,1633.80,0.0,0.0,0.0,0.0,0.0,1.0,A,0.0,1.0,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,F1,2000-07-29,NaN,Auxerre,Sedan,1635.58,1624.22,0.0,0.0,0.0,0.0,0.0,1.0,A,0.0,1.0,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.70,3.3,3.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,F1,2000-07-29,NaN,Bordeaux,Metz,1734.34,1673.11,0.0,0.0,0.0,0.0,1.0,1.0,D,1.0,0.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.65,3.3,4.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,date,club,country,elo
0,2000-07-01,Aachen,GER,1453.60
1,2000-07-01,Aalborg,DEN,1482.61
2,2000-07-01,Aalst,BEL,1337.53
3,2000-07-01,Aarhus,DEN,1381.46
4,2000-07-01,Aberdeen,SCO,1360.43


## 2. 移除高缺失率的项

`HomeShots` 至 `C_PHB` 包含技术统计、额外盘口与 `C_*` 派生列，缺失率明显高于身份与比分列。


In [2]:
i0, i1 = raw.columns.get_loc('HomeShots'), raw.columns.get_loc('C_PHB')
drop = [c for c in raw.columns[i0:i1 + 1] if c not in ODDS] + ['MatchTime']
raw = raw.drop(columns=drop)
print('dropped', drop)
print(raw.shape, list(raw.columns))


dropped ['HomeShots', 'AwayShots', 'HomeTarget', 'AwayTarget', 'HomeFouls', 'AwayFouls', 'HomeCorners', 'AwayCorners', 'HomeYellow', 'AwayYellow', 'HomeRed', 'AwayRed', 'MaxHome', 'MaxDraw', 'MaxAway', 'Over25', 'Under25', 'MaxOver25', 'MaxUnder25', 'HandiSize', 'HandiHome', 'HandiAway', 'C_LTH', 'C_LTA', 'C_VHD', 'C_VAD', 'C_HTB', 'C_PHB', 'MatchTime']
(238858, 19) ['Division', 'MatchDate', 'HomeTeam', 'AwayTeam', 'HomeElo', 'AwayElo', 'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'FTHome', 'FTAway', 'FTResult', 'HTHome', 'HTAway', 'HTResult', 'OddHome', 'OddDraw', 'OddAway']


## 3. 比分缺失行

`Division`、`MatchDate`、`HomeTeam`、`AwayTeam` 无缺失。`FTHome`、`FTAway`、`FTResult` 同时缺失的比赛无法作为目标变量，删除这些行，不填补赛果。


In [3]:
print(raw[CORE].isna().sum().to_string())
display(raw.loc[raw[CORE].isna().any(axis=1), CORE])

matches = raw.copy()
matches.insert(0, 'SourceRow', range(2, len(matches) + 2))
matches = matches.dropna(subset=CORE)
for c in ['Division', 'HomeTeam', 'AwayTeam']:
    matches[c] = matches[c].map(norm)
matches['MatchDate'] = pd.to_datetime(matches.MatchDate)
matches['FTHome'] = pd.to_numeric(matches.FTHome)
matches['FTAway'] = pd.to_numeric(matches.FTAway)
exp = np.where(matches.FTHome > matches.FTAway, 'H', np.where(matches.FTHome < matches.FTAway, 'A', 'D'))
matches = matches.loc[matches.FTResult.eq(exp) & matches.HomeTeam.ne(matches.AwayTeam)].copy()
matches['Country'] = matches.Division.map(COUNTRY)
matches = matches.sort_values(['MatchDate', 'SourceRow']).reset_index(drop=True)
print('kept', matches.shape)


Division     0
MatchDate    0
HomeTeam     0
AwayTeam     0
FTHome       4
FTAway       4
FTResult     4


,Division,MatchDate,HomeTeam,AwayTeam,FTHome,FTAway,FTResult
112580,G1,2015-05-10,Niki Volos,OFI,NaN,NaN,NaN
130377,BRA,2016-11-12,Chapecoense-SC,Atletico-MG,NaN,NaN,NaN
159189,G1,2019-03-17,Panathinaikos,Olympiakos,NaN,NaN,NaN
233555,F2,2025-12-05,Bastia,Red Star,NaN,NaN,NaN


kept (238854, 21)


## 4. Elo 缺失与补全

`HomeElo` / `AwayElo` 存在大量缺失。`EloRatings` 给出同一国家、同一球队、按日期的评分，用于填补 Matches 中空缺的一侧。

**规则**
- 原表已有 Elo 的不覆盖。
- 缺失侧按国家与规范化队名匹配，取严格早于比赛日的最近一条。
- 不使用当天或未来评分。
- 补全后仍缺任一侧的比赛删除。

Form 历史基于全部有效赛果计算，包括因 Elo 仍缺而未导出的场次。


In [4]:
print(matches[['HomeElo', 'AwayElo']].isna().sum())

elo['date'] = pd.to_datetime(elo.date)
elo['elo'] = pd.to_numeric(elo.elo, errors='coerce')
elo['club'] = elo.club.map(norm)
elo['country'] = elo.country.map(norm)
elo = (elo.dropna(subset=['date', 'club', 'country', 'elo'])
         .drop_duplicates(['country', 'club', 'date'])
         .sort_values('date'))

history = matches.copy()
for side in ['Home', 'Away']:
    col = side + 'Elo'
    history[col] = pd.to_numeric(history[col], errors='coerce')
    miss = history[col].isna()
    left = (history.loc[miss, ['MatchDate', 'Country', side + 'Team']]
            .rename(columns={side + 'Team': 'club', 'Country': 'country'})
            .assign(row=lambda d: d.index)
            .sort_values('MatchDate'))
    got = pd.merge_asof(left, elo, left_on='MatchDate', right_on='date',
                        by=['country', 'club'], direction='backward',
                        allow_exact_matches=False)
    history.loc[got.row, col] = got.elo.to_numpy()
    print(col, 'missing before fill:', int(miss.sum()),
          '| unmatched after fill:', int(history[col].isna().sum()))

have_elo = history[['HomeElo', 'AwayElo']].notna().all(axis=1)
print('rows dropped (Home or Away Elo still NA):', int((~have_elo).sum()))


HomeElo    88959
AwayElo    89028
dtype: int64
HomeElo missing before fill: 88959 | unmatched after fill: 78336
AwayElo missing before fill: 89028 | unmatched after fill: 78369
rows dropped (Home or Away Elo still NA): 92951


## 5. 由数据得出 Form 公式并重算

原始 `Form3` / `Form5` 定义不明确：赛季初常为 0，且无法从当前行还原所用场次。根据 `FTResult` 按足球积分重算赛前状态。

**积分**

$$
\text{points} =
\begin{cases}
3 & \text{该队胜} \\
1 & \text{平} \\
0 & \text{负}
\end{cases}
$$

**Form**

$$
\mathrm{Form}k = \sum_{j=1}^{k} \text{points}_{t-j}
$$

- $k=3$ 或 $5$；必须恰好 $k$ 场才写数，否则为缺失
- $\text{points}_{t-j}$ 只来自**严格早于本场日期**的比赛
- 不含本场、不含同日
- 按国家 + 球队连贯历史（含升降级）

双方 `Form5` 必须均有完整近 5 场；任一不足则删除该场，以保证建模样本的赛前状态长度一致。


In [5]:
past = defaultdict(lambda: deque(maxlen=5))
form = {c: np.full(len(history), np.nan) for c in ['Form3Home', 'Form5Home', 'Form3Away', 'Form5Away']}
for day, batch in history.groupby('MatchDate', sort=True):
    for r in batch.itertuples():
        for side in ['Home', 'Away']:
            prior = list(past[(r.Country, getattr(r, side + 'Team'))])
            for k in (3, 5):
                if len(prior) >= k:
                    form[f'Form{k}{side}'][r.Index] = sum(prior[-k:])
    for r in batch.itertuples():
        for side, win in [('Home', 'H'), ('Away', 'A')]:
            pts = 1 if r.FTResult == 'D' else 3 if r.FTResult == win else 0
            past[(r.Country, getattr(r, side + 'Team'))].append(pts)

for c, v in form.items():
    history[c] = pd.array(v, dtype='Int64')
completed = history.loc[have_elo].copy()
form5_ok = completed[['Form5Home', 'Form5Away']].notna().all(axis=1)
print('dropped incomplete Form5', int((~form5_ok).sum()))
completed = completed.loc[form5_ok].reset_index(drop=True)
display(completed[['MatchDate', 'HomeTeam', 'AwayTeam', 'FTResult',
                   'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away']].head())


dropped incomplete Form5 2170


,MatchDate,HomeTeam,AwayTeam,FTResult,Form3Home,Form5Home,Form3Away,Form5Away
0,2000-08-29,Cannes,Angers,H,2,6,1,2
1,2000-08-29,Creteil,Chateauroux,A,6,10,4,5
2,2000-08-29,Gueugnon,Caen,D,5,5,0,1
3,2000-08-29,Laval,Nice,H,6,9,4,8
4,2000-08-29,Le Havre,Ajaccio,H,7,11,1,5


## 6. 检查 OddHome / OddDraw / OddAway

小数赔率应为有限数且大于 1。缺失或 `<= 1` 的行删除。


In [6]:
odds = completed[ODDS].apply(pd.to_numeric, errors='coerce')
bad = odds.isna().any(axis=1) | odds.le(1).any(axis=1)
print('missing or <= 1:', int(bad.sum()))
display(completed.loc[bad, ['MatchDate', 'HomeTeam', 'AwayTeam'] + ODDS].head())

cleaned = completed.loc[~bad].copy()
cleaned[ODDS] = odds.loc[~bad]
cleaned = cleaned.reset_index(drop=True)
print('kept', cleaned.shape)


missing or <= 1: 2000


,MatchDate,HomeTeam,AwayTeam,OddHome,OddDraw,OddAway
0,2000-08-29,Cannes,Angers,NaN,NaN,NaN
1,2000-08-29,Creteil,Chateauroux,NaN,NaN,NaN
2,2000-08-29,Gueugnon,Caen,NaN,NaN,NaN
3,2000-08-29,Laval,Nice,NaN,NaN,NaN
4,2000-08-29,Le Havre,Ajaccio,NaN,NaN,NaN


kept (141733, 21)


## 7. 保存

双方近 5 场不足的比赛已删除。结果写入 `data/processed/cleaned.csv`。


In [7]:
out = ROOT / 'data/processed/cleaned.csv'
out.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(out, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
print(n_raw, 'raw ->', len(matches), 'valid score ->', len(completed), 'both Elo ->', len(cleaned), 'cleaned')
print('Wrote', out)
display(cleaned.head())


238858 raw -> 238854 valid score -> 143733 both Elo -> 141733 cleaned
Wrote D:\code\aml_coursework\data\processed\cleaned.csv


,SourceRow,Division,MatchDate,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult,HTHome,HTAway,HTResult,OddHome,OddDraw,OddAway,Country
0,411,F1,2000-09-05,Monaco,Marseille,1766.17,1678.53,7,8,4,7,0.0,2.0,A,0.0,0.0,D,1.80,3.1,4.0,FRA
1,438,F1,2000-09-06,Bastia,Lille,1671.73,1660.00,3,9,6,10,1.0,0.0,H,0.0,0.0,D,1.66,3.2,4.4,FRA
2,439,F1,2000-09-06,Guingamp,Toulouse,1560.95,1547.54,0,1,2,3,2.0,1.0,H,0.0,1.0,A,2.15,3.0,3.1,FRA
3,440,F1,2000-09-06,Lens,Auxerre,1751.70,1643.20,5,11,6,9,1.0,0.0,H,1.0,0.0,H,1.75,3.2,4.1,FRA
4,441,F1,2000-09-06,Metz,Strasbourg,1671.09,1605.71,4,5,3,3,1.0,0.0,H,0.0,0.0,D,1.71,3.0,4.9,FRA
